In [2]:
# Import requirements
from dotenv import load_dotenv
import anthropic
import numpy as np

In [3]:
# loading the environment variables from the .env file
load_dotenv()

True

#### Basics of sending queries to the Anthropic API

In [8]:
client = anthropic.Anthropic()

message = client.messages.create(
    model='claude-haiku-4-5',
    max_tokens=10000,
    messages=[
        {
            "role": "user",
            "content": "What should I search for to find the latest developments in renewable energy?",
        }
    ],
)
print(message.content)

[TextBlock(citations=None, text='# Search Strategies for Renewable Energy News\n\n**Specific search terms:**\n- "renewable energy news 2024"\n- "solar/wind/battery technology breakthroughs"\n- "clean energy policy updates"\n- "renewable energy stocks" (if investment-focused)\n\n**Reliable sources to check directly:**\n- RenewableEnergyWorld.com\n- PV Magazine\n- Windpower Monthly\n- Carbon Brief\n- IEA (International Energy Agency) reports\n- Your country\'s energy department\n\n**Broader searches:**\n- "sustainable energy developments"\n- "net zero technology"\n- "electric vehicle charging infrastructure"\n\n**By region/focus:**\n- "[Your country] renewable energy targets"\n- "green hydrogen projects"\n- "offshore wind farms"\n\n**Pro tips:**\n- Set up Google News alerts for specific topics\n- Follow industry publications or research institutes\n- Search for reports from organizations like IRENA or NREL\n- Check government energy ministry websites for policy changes\n\nWhat specific a

#### Basics of creating an agent

This agent will run simulations by running a bunch of bernoulli experiements and summarizing the results.

It will only have two tools: run_bernoulli_experiments and summarize_results.

In [4]:
# Tool Definitions
def run_bernoulli_experiments(probability, runs):
    return np.random.binomial(n=1, p=probability, size=runs)

def summarize_results(input_array, method="mean"):
    if method == "mean":
        return np.mean(input_array)
    elif method == "median":
        return np.median(input_array)
    else:
        raise f"Error: unsupported method '{method}'. Choose 'mean' or 'median'."

In [5]:
# Defining the system prompt

SYSTEM_PROMPT="""
You are a fan of probability and like to help people understand the impact of biased coins.

If someone asks you to run a simulation, please use the run_bernoulli_experiments tool. 
This tool takes two arguments: probability and runs. These are required, make sure the user
gives you inputs. If the person doesn't give you inputs, ask them for the probability of heads 
and the number of runs to complete.

After running a summary, ask the user if they would like a summary. If they say yes, ask them
if they would like to use the mean or median to summarize before running the summarize_results tool. 

After summarizing the results, explain the results in plain language. If a summary is not requested,
explain the results in plain language without summarizing.
"""

In [9]:
TOOL_REGISTRY = {
    "run_bernoulli_experiments": run_bernoulli_experiments,
    "summarize_results": summarize_results
}

TOOL_DEFINITIONS = [
    {
        "name": "run_bernoulli_experiments",
        "description": "Runs a series of bernoulli trials and returns the results",
        "input_schema":{
            "type": "object",
            "properties": {
                "probability": {
                    "type": "number",
                    "description": "probability of heads on each trial, between 0 and 1"
                },
                "runs": {
                    "type": "integer",
                    "description": "Numer of Bernoulli trials to run."
                }
            },
            "required": ["probability", "runs"]
        }
    },
    {
        "name": "summarize_results",
        "description": "Summarizes the results from the Bernoulli trials",
        "input_schema": {
            "type": "object",
            "properties": {
                "input_array": {
                    "type": "array",
                    "items": {"type": "number"},
                    "description": "the array of data to summarize"
                },
                "method": {
                    "type": "string",
                    "description": "statistical method to apply to summarize results"
                }
            },
            "required": ["input_array"]
        }
    }
]

In [12]:
def run_agent(user_input: str):
    client = anthropic.Anthropic()
    messages = [{"role": "user", "content": user_input}]

    MAX_ITERATIONS = 10
    iteration = 0

    while iteration <= MAX_ITERATIONS:
        iteration += 1
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=TOOL_DEFINITIONS,
            messages=messages
        )

        # Append the assistant's response to history
        messages.append({"role": "assistant", "content": response.content})

        # If the model is done, print and exit
        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    print(f"Agent: {block.text}")
            break

        # If the model wants to use a tool
        if response.stop_reason == "tool_use":
            tool_results = []

            for block in response.content:
                if block.type == "tool_use":
                    tool_name = block.name
                    tool_args = block.input

                    print(f"[Calling tool: {tool_name} with {tool_args}]")

                    # Dispatch to the right function via the registry
                    if tool_name in TOOL_REGISTRY:
                        result = TOOL_REGISTRY[tool_name](**tool_args)
                    else:
                        result = f"Error: unknown tool '{tool_name}'"

                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })

            # Append tool results and loop back
            messages.append({"role": "user", "content": tool_results})


In [13]:
run_agent("Can you run a simulation with probability 0.3 with 100 runs? Summarize the results using the mean.")

[Calling tool: run_bernoulli_experiments with {'probability': 0.3, 'runs': 100}]
[Calling tool: summarize_results with {'input_array': [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], 'method': 'mean'}]
Agent: Here's a plain-language breakdown of the results:

- **Simulation**: We flipped a biased coin 100 times, where the probability of getting heads (1) on any given flip was **0.3** (or 30%).
- **Outcome**: Out of 100 flips, we got **24 heads** and **76 tails**.
- **Mean (Average)**: The mean of the results was **0.24**, meaning heads came up 24% of the time.

This is pretty close to the expected probability of 0.3 (30%), but not exactly — and that's completely normal! With only 100 trials, we expect some natural **random variation**. 